Sections:
  1. Setup & Configuration
  2. Data Module
  3. Physics Engine Module
  4. Metrics Module
  5. Optimization Module
  6. Evaluation Module
  7. Visualization Module
  8. Checkpointing & Reporting Module
  9. Main Pipeline

#  1. Setup & Configuration

## 1.1 Taking care of libraries

In [ ]:
!pip install uv

In [ ]:
!uv pip install numpy pandas torch pytorch-lightning matplotlib scipy tqdm optuna seaborn dcor xlrd sdmetrics kaleido==0.2.1 pydrive2 polars fastexcel

In [ ]:
import sys
import os
import time
import json
import resource
import gc
from itertools import combinations
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.ndimage import uniform_filter1d
from scipy.spatial.distance import cdist, jensenshannon
from scipy.stats import wasserstein_distance
from sklearn.metrics import mutual_info_score
from sklearn.preprocessing import StandardScaler

import optuna
from optuna.samplers import TPESampler
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances

from sdmetrics.reports.single_table import DiagnosticReport, QualityReport
from sdmetrics.single_table import (
    ContingencySimilarity, CorrelationSimilarity,
    DCRBaselineProtection, KSComplement, TVComplement,
)
from sdmetrics.visualization import get_column_pair_plot, get_column_plot

import gdown
import tqdm as tqdm_mod

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import seaborn as sns

## 1.2 Configurations

In [ ]:
_WALL_CLOCK = {}
_PEAK_MEM = {}

def _rss_mb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024

def _snap(label, t0, m0):
    _WALL_CLOCK[label] = time.time() - t0
    _PEAK_MEM[label] = _rss_mb() - m0

RANDOM_SEEDS = [
    776, 578, 2024, 678, 1011, 1415, 982, 807, 521, 2357,
    4271, 4069, 289, 675, 3333, 777, 4321, 991, 1213, 1453,
]

OPTUNA_SEEDS = [42, 0, 1, 2, 3]
OPTUNA_N_TRIALS = 500
OPTUNA_TIMEOUT = 3600000

PHYSICAL_BOUNDS = {
    "GR": (0, 200), "DT": (30, 180), "RHOB_combined": (1.2, 2.9),
    "RES_DEEP": (0.01, 10_000), "HP": (500, 15_000), "OB": (2_000, 40_000),
    "DT_NCT": (30, 180), "PPP": (0, 30_000),
}

SENTINEL_VALUES = {-999, -999.25, -999.0}

In [ ]:
GDRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/PhysicsBenchmark"

In [ ]:
GDRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    GDRIVE_MOUNTED = True
    print(" Google Drive mounted — checkpoints will be saved to Drive.")
except Exception:
    print("ℹ Google Drive not available — checkpoints will be saved locally only.")

In [ ]:
REGION_CONFIGS = {
    "r1": {
        "name": "Region 1 — Missa Keswal, Eastern Potwar Basin, Pakistan",
        "tag": "r1",
        "mode": "full",  # 8-column: includes HP, OB, DT_NCT, PPP
        "file_ids": [
            "1ItU6vXk2saTi4ofGo95OHjr39qgsXTan",
            "1Q0io7g2-K9ALS8enQlFRM1RdI-H1vQQ3",
            "1Lu-JtVdPckJ2l08fVB8ZMKrdKAMhxLdt",
        ],
        "file_names": [
            "MISSA-KESWAL-01.csv", "MISSA-KESWAL-02.csv", "MISSA-KESWAL-03.csv",
        ],
        "well_names": ["Well_1", "Well_2", "Well_3"],
        "file_type": "csv",
        "log_cols": ["GR", "DT", "RHOB_combined", "RES_DEEP", "HP", "OB", "DT_NCT", "PPP"],
        "initial_hp": {
            "phi0": 0.45, "compaction_c": 0.00040, "phi_noise_std": 0.025,
            "phi_layer_amp": 0.04, "phi_layer_period": 35.0,
            "dt_matrix": 55.5, "dt_noise_std": 3.0, "dt_fluid": 189.0,
            "rho_matrix": 2.65, "rho_fluid": 1.03, "rhob_noise_std": 0.02,
            "archie_a": 1.0, "archie_m": 2.0, "archie_n": 2.0,
            "Rw": 0.05, "Sw": 0.85, "res_log_noise_std": 0.05,
            "res_clip_min": 0.01, "res_clip_max": 10_000,
            "gr_clean": 20.0, "gr_shale": 130.0, "gr_noise_std": 8.0,
            "hp_fluid_density": 1.03, "psi_per_ft_per_gcc": 1.4223,
            "ob_shallow_rhob": 2.30,
            "nct_dt0": 50.0, "nct_dt_surface": 200.0, "nct_k": 0.0012,
            "nct_smooth_window": 200, "nct_weight_smooth": 0.90,
            "eaton_n": 3.0,
        },
        "pairs_of_interest": [["DT", "RHOB_combined"], ["GR", "DT"], ["GR", "RHOB_combined"]],
    },
    "r2": {
        "name": "Region 2 — Pindori, Eastern Potwar Basin, Pakistan",
        "tag": "r2",
        "mode": "full",
        "file_ids": [
            "1yvCAJaRbf_rL9wGzE3KJ39j5HaGi1q6g",
            "1HVFJx2aHu1xf57fgd8WZXDZ79MpPQLY-",
            "1qOdM_8DLwhJlJc9F6Dn-Z4btoYhgVDH1",
        ],
        "file_names": ["/content/PINDORI-1.csv", "/content/PINDORI-2.csv", "/content/PINDORI-3.csv"],
        "well_names": ["Pindori_1", "Pindori_2", "Pindori_3"],
        "file_type": "csv",
        "log_cols": ["GR", "DT", "RHOB_combined", "RES_DEEP", "HP", "OB", "DT_NCT", "PPP"],
        "initial_hp": {
            "phi0": 0.45, "compaction_c": 0.00040, "phi_noise_std": 0.025,
            "phi_layer_amp": 0.04, "phi_layer_period": 35.0,
            "dt_matrix": 55.5, "dt_noise_std": 3.0, "dt_fluid": 189.0,
            "rho_matrix": 2.65, "rho_fluid": 1.03, "rhob_noise_std": 0.02,
            "archie_a": 1.0, "archie_m": 2.0, "archie_n": 2.0,
            "Rw": 0.05, "Sw": 0.85, "res_log_noise_std": 0.05,
            "res_clip_min": 0.01, "res_clip_max": 10_000,
            "gr_clean": 20.0, "gr_shale": 130.0, "gr_noise_std": 8.0,
            "hp_fluid_density": 1.03, "psi_per_ft_per_gcc": 1.4223,
            "ob_shallow_rhob": 2.30,
            "nct_dt0": 50.0, "nct_dt_surface": 200.0, "nct_k": 0.0012,
            "nct_smooth_window": 200, "nct_weight_smooth": 0.90,
            "eaton_n": 3.0,
        },
        "pairs_of_interest": [["DT", "RHOB_combined"], ["GR", "DT"], ["GR", "RHOB_combined"]],
    },
    "r3": {
        "name": "Region 3 — Joyamair/Minwal, Eastern Potwar Basin, Pakistan",
        "tag": "r3",
        "mode": "full",
        "file_ids": [
            "1bvH0gDMdd3ZG15HnWXmzFQPqpfjxEspn",
            "1hjJvfDUEf_LzZFJPohUmzV96UGuPlVPr",
            "1X8Pj7ukoXDXcRGg3swyuWVFgaBeg4qbb",
        ],
        "file_names": [
            "/content/JOYAMAIR-4.csv", "/content/MINWAL-2.csv", "/content/MINWAL-X-1.csv",
        ],
        "well_names": ["Joyamair_4", "Minwal_2", "JM_3"],
        "file_type": "csv",
        "log_cols": ["GR", "DT", "RHOB_combined", "RES_DEEP", "HP", "OB", "DT_NCT", "PPP"],
        "initial_hp": {
            "phi0": 0.45, "compaction_c": 0.00040, "phi_noise_std": 0.025,
            "phi_layer_amp": 0.04, "phi_layer_period": 35.0,
            "dt_matrix": 55.5, "dt_noise_std": 3.0, "dt_fluid": 189.0,
            "rho_matrix": 2.65, "rho_fluid": 1.03, "rhob_noise_std": 0.02,
            "archie_a": 1.0, "archie_m": 2.0, "archie_n": 2.0,
            "Rw": 0.05, "Sw": 0.85, "res_log_noise_std": 0.05,
            "res_clip_min": 0.01, "res_clip_max": 10_000,
            "gr_clean": 20.0, "gr_shale": 130.0, "gr_noise_std": 8.0,
            "hp_fluid_density": 1.03, "psi_per_ft_per_gcc": 1.4223,
            "ob_shallow_rhob": 2.30,
            "nct_dt0": 50.0, "nct_dt_surface": 200.0, "nct_k": 0.0012,
            "nct_smooth_window": 200, "nct_weight_smooth": 0.90,
            "eaton_n": 3.0,
        },
        "pairs_of_interest": [["DT", "RHOB_combined"], ["GR", "DT"], ["GR", "RHOB_combined"]],
    },
    "r4": {
        "name": "Region 4 — IODP Expedition 323, Bering Sea",
        "tag": "r4",
        "mode": "basic",  # 4-column: GR, DT, RHOB, RES only
        "file_ids": ["1ZNn5wgI6ODe-VV4gZGp-HtUj6mY0bJkP"],
        "file_names": ["merged_logs.xlsx"],
        "well_names": ["U1343E"],
        "file_type": "xlsx",
        "xlsx_slice": (677, 4755),
        "xlsx_col_map": {
            "DEPTH_m": "DEPTH", "GR_gAPI": "GR", "DT_us_ft": "DT",
            "RHOB_g_cm3": "RHOB_combined", "RES_DEEP_ohmm": "RES_DEEP",
        },
        "log_cols": ["GR", "DT", "RHOB_combined", "RES_DEEP"],
        "initial_hp": {
            "phi0": 0.72, "compaction_c": 0.00005, "phi_noise_std": 0.015,
            "phi_layer_amp": 0.03, "phi_layer_period": 3.0,
            "dt_matrix": 108.0, "dt_noise_std": 0.5, "dt_fluid": 189.0,
            "rho_matrix": 2.65, "rho_fluid": 1.025, "rhob_noise_std": 0.02,
            "archie_a": 1.0, "archie_m": 1.5, "archie_n": 2.0,
            "Rw": 0.41, "Sw": 1.0, "res_log_noise_std": 0.06,
            "res_clip_min": 0.01, "res_clip_max": 10_000,
            "gr_clean": 48.5, "gr_shale": 80.0, "gr_noise_std": 1.5,
        },
        "pairs_of_interest": [
            ["DT", "RHOB_combined"], ["GR", "DT"],
            ["GR", "RHOB_combined"], ["GR", "RES_DEEP"],
        ],
    },
    "r5": {
        "name": "Region 5 — Volve Oil Well, North Sea",
        "tag": "r5",
        "mode": "basic",
        "file_ids": [
            "1aNypmAzJ3DDrwuGIGcEt0vFrv_mt0QBL8Y6nrmfd-aM",
            "1kNmd-twgBwAXhg4PHHzFiCEvyeWLg5NApi4Afy5CZeE",
            "1ljgLkr-yAVPvxMli083cPdYdEEhfLeSbORlFdCFfeGg",
            "1dRLdtFhT5VjL3ZJvQQRsL5AFhJCgJaVoY0LMBHS2v0M",
            "13W4pcZpn-hrtM9BBVci_FKVzHQQCLudB_dJ_z4K6-g0",
        ],
        "file_names": ["F-11A.xlsx", "F-11T2.xlsx", "F-12.xlsx", "F-1A.xlsx", "F-1B.xlsx"],
        "well_names": ["F-11A", "F-11T2", "F-12", "F-1A", "F-1B"],
        "file_type": "volve",
        "log_cols": ["GR", "DT", "RHOB_combined", "RES_DEEP"],
        "initial_hp": {
            "phi0": 0.40, "compaction_c": 0.0004, "phi_noise_std": 0.025,
            "phi_layer_amp": 0.04, "phi_layer_period": 35.0,
            "dt_matrix": 55.5, "dt_noise_std": 3.0, "dt_fluid": 189.0,
            "rho_matrix": 2.65, "rho_fluid": 1.03, "rhob_noise_std": 0.02,
            "archie_a": 1.0, "archie_m": 2.0, "archie_n": 2.0,
            "Rw": 0.05, "Sw": 0.85, "res_log_noise_std": 0.05,
            "res_clip_min": 0.01, "res_clip_max": 10_000,
            "gr_clean": 20.0, "gr_shale": 130.0, "gr_noise_std": 8.0,
        },
        "pairs_of_interest": [
            ["DT", "RHOB_combined"], ["GR", "DT"],
            ["GR", "RHOB_combined"], ["GR", "RES_DEEP"],
        ],
    },
}

# 2. Data Module

## 2.1 Downloading and cleaning the data

In [ ]:
def download_files(file_ids):
    for fid in file_ids:
        url = f"https://drive.google.com/uc?id={fid}"
        gdown.download(url, quiet=False)

In [ ]:
def clean_well_data(df, outlier_std=5, verbose=True, label=""):
    df = df.copy()
    n_before = len(df)
    if "SPHI" in df.columns:
        df.drop(columns=["SPHI"], inplace=True)

    log_cols = [c for c in df.columns if c != "DEPTH"]

    for col in log_cols:
        df.loc[df[col].isin(SENTINEL_VALUES), col] = np.nan

    for col in log_cols:
        if col in PHYSICAL_BOUNDS:
            lo, hi = PHYSICAL_BOUNDS[col]
            df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan

    for col in log_cols:
        clean_vals = df[col].dropna()
        if len(clean_vals) < 10:
            continue
        mu, sig = clean_vals.mean(), clean_vals.std()
        if sig == 0:
            continue
        df.loc[(df[col] - mu).abs() > outlier_std * sig, col] = np.nan

    df.dropna(subset=log_cols, how="all", inplace=True)
    df.reset_index(drop=True, inplace=True)

    if verbose:
        removed = n_before - len(df)
        pct = 100 * removed / n_before if n_before else 0
        print(f"  [{label}] {n_before} → {len(df)} rows (removed {removed}, {pct:.1f}%)")
    return df

## 2.2 Loading the data

In [ ]:
def load_region_data(cfg):
    """Load and clean real well data for a region config."""
    download_files(cfg["file_ids"])

    if cfg["file_type"] == "csv":
        dfs = []
        for fp in cfg["file_names"]:
            df = pd.read_csv(fp)
            # Normalise column names
            col_map = {c: "RHOB_combined" for c in df.columns
                       if c.upper() == "RHOB_COMBINED" and c != "RHOB_combined"}
            if col_map:
                df = df.rename(columns=col_map)
            dfs.append(df)
        # Clean
        cleaned = []
        for name, df in zip(cfg["well_names"], dfs):
            cleaned.append(clean_well_data(df, label=f"{name} REAL"))
        return cleaned

    elif cfg["file_type"] == "xlsx":
        df = pd.read_excel(cfg["file_names"][0])
        sl = cfg.get("xlsx_slice")
        if sl:
            df = df.iloc[sl[0]:sl[1]]
        col_map = cfg.get("xlsx_col_map", {})
        if col_map:
            df = df.rename(columns=col_map)
        # Coerce every column to numeric — Excel often leaves object dtype
        for c in df.columns:
           df[c] = pd.to_numeric(df[c], errors="coerce")
        df = df.dropna()
        df = df.reset_index(drop=True)
        return [df]

    elif cfg["file_type"] == "volve":
        import polars as pl
        dfs = []
        for fp in cfg["file_names"]:
            df_pol = pl.read_excel(fp).drop_nulls()
            df_pd = df_pol.to_pandas()
            rename_map = {}
            for c in df_pd.columns:
                cu = c.upper().strip()
                if cu in ("RHOB", "RHOB_COMBINED", "RHOZ"):
                    rename_map[c] = "RHOB_combined"
                elif cu in ("RES_DEEP", "RDEP", "RT", "ILD", "RDEEP", "LLD"):
                    rename_map[c] = "RES_DEEP"
                elif cu == "GR":
                    rename_map[c] = "GR"
                elif cu in ("DT", "DTC", "DTCO"):
                    rename_map[c] = "DT"
                elif cu in ("DEPTH", "DEPT"):
                    rename_map[c] = "DEPTH"
            df_pd = df_pd.rename(columns=rename_map)
            for col in df_pd.columns:
                df_pd[col] = pd.to_numeric(df_pd[col], errors="coerce")
            df_pd = df_pd.dropna().reset_index(drop=True)
            dfs.append(df_pd)
        return dfs

    raise ValueError(f"Unknown file_type: {cfg['file_type']}")

#   3. Physics Engine Module

## 3.1 Defining physics formulas

In [ ]:
def make_porosity(depth, seed, hp):
    rng = np.random.default_rng(seed)
    trend = hp["phi0"] * np.exp(-hp["compaction_c"] * depth)
    layers = hp["phi_layer_amp"] * np.sin(depth / hp["phi_layer_period"])
    noise = rng.normal(0, hp["phi_noise_std"], len(depth))
    return np.clip(trend + layers + noise, 0.02, 0.50)

In [ ]:
def phi_to_dt(phi, seed, hp):
    rng = np.random.default_rng(seed)
    dt = phi * hp["dt_fluid"] + (1 - phi) * hp["dt_matrix"]
    dt += rng.normal(0, hp["dt_noise_std"], len(phi))
    return np.clip(dt, *PHYSICAL_BOUNDS["DT"])

def phi_to_rhob(phi, depth, seed, hp):
    rng = np.random.default_rng(seed)
    lith_var = 0.1 * np.sin(depth / 120)
    rho_matrix_var = hp["rho_matrix"] + lith_var
    rhob = phi * hp["rho_fluid"] + (1 - phi) * rho_matrix_var
    rhob += rng.normal(0, hp["rhob_noise_std"], len(phi))
    return np.clip(rhob, *PHYSICAL_BOUNDS["RHOB_combined"])

def phi_to_res(phi, seed, hp):
    rng = np.random.default_rng(seed)
    # Guard phi against underflow: small phi + large archie_m → Rt explodes
    phi_safe = np.clip(phi, 0.05, 0.50)
    Rt = hp["archie_a"] * hp["Rw"] / (phi_safe ** hp["archie_m"] * hp["Sw"] ** hp["archie_n"])
    log_Rt = np.log10(Rt) + rng.normal(0, hp["res_log_noise_std"], len(phi))
    return np.clip(10 ** log_Rt, *PHYSICAL_BOUNDS["RES_DEEP"])

def phi_to_gr(phi, seed, hp):
    rng = np.random.default_rng(seed)
    phi_norm = np.clip((phi - 0.02) / (0.50 - 0.02), 0, 1)
    gr = hp["gr_shale"] + phi_norm * (hp["gr_clean"] - hp["gr_shale"])
    gr += rng.normal(0, hp["gr_noise_std"], len(phi))
    return np.clip(gr, *PHYSICAL_BOUNDS["GR"])



In [ ]:
def calc_hp(depth, hp):
    return hp["psi_per_ft_per_gcc"] * hp["hp_fluid_density"] * depth

def calc_ob(depth, rhob, hp):
    c = hp["psi_per_ft_per_gcc"]
    ob = np.zeros_like(depth, dtype=float)
    ob[0] = hp["ob_shallow_rhob"] * c * depth[0]
    rho_avg = (rhob[:-1] + rhob[1:]) / 2
    dz = np.gradient(depth[:-1])
    ob[1:] = ob[0] + np.cumsum(rho_avg * c * dz)
    return ob

def calc_dt_nct(depth, dt, hp):
    depth = np.asarray(depth, dtype=float)
    dt    = np.asarray(dt, dtype=float)

    # Physical exponential trend from HPs (Optuna-searched)
    trend = hp["nct_dt0"] + (hp["nct_dt_surface"] - hp["nct_dt0"]) * \
            np.exp(-hp["nct_k"] * depth)

    # Smoothed observed DT — captures the actual compaction shape
    win = int(max(3, min(hp["nct_smooth_window"], len(dt))))
    dt_smooth = uniform_filter1d(dt, size=win, mode="nearest")

    # Blend trend (physics prior) with smoothed observation
    w = float(np.clip(hp["nct_weight_smooth"], 0.0, 1.0))
    dt_nct = w * dt_smooth + (1.0 - w) * trend

    return np.clip(dt_nct, *PHYSICAL_BOUNDS["DT_NCT"])

def calc_ppp(dt, dt_nct, ob, hydro, hp):
    return ob - (ob - hydro) * (dt / dt_nct) ** hp["eaton_n"]

## 3.2 Defining syntethic data

In [ ]:
def build_synthetic(seed, hp, depth, mode="full"):
    """
    Build one complete synthetic dataset.
    mode='full'  → 8 columns (GR, DT, RHOB, RES, HP, OB, DT_NCT, PPP)
    mode='basic' → 4 columns (GR, DT, RHOB, RES)
    """
    phi  = make_porosity(depth, seed, hp)
    dt   = phi_to_dt(phi, seed + 10, hp)
    rhob = phi_to_rhob(phi, depth, seed + 20, hp)
    res  = phi_to_res(phi, seed + 30, hp)
    gr   = phi_to_gr(phi, seed + 40, hp)

    result = {"DEPTH": depth, "GR": gr, "DT": dt, "RHOB_combined": rhob, "RES_DEEP": res}

    if mode == "full":
        hydro  = calc_hp(depth, hp)
        ob     = calc_ob(depth, rhob, hp)
        dt_nct = calc_dt_nct(depth, dt, hp)
        ppp    = calc_ppp(dt, dt_nct, ob, hydro, hp)

        gr     = np.clip(gr,     *PHYSICAL_BOUNDS["GR"])
        dt     = np.clip(dt,     *PHYSICAL_BOUNDS["DT"])
        rhob   = np.clip(rhob,   *PHYSICAL_BOUNDS["RHOB_combined"])
        res    = np.clip(res,    *PHYSICAL_BOUNDS["RES_DEEP"])
        hydro  = np.clip(hydro,  *PHYSICAL_BOUNDS["HP"])
        ob     = np.clip(ob,     *PHYSICAL_BOUNDS["OB"])
        dt_nct = np.clip(dt_nct, *PHYSICAL_BOUNDS["DT_NCT"])
        ppp    = np.clip(ppp,    *PHYSICAL_BOUNDS["PPP"])

        result.update({
            "GR": gr, "DT": dt, "RHOB_combined": rhob, "RES_DEEP": res,
            "HP": hydro, "OB": ob, "DT_NCT": dt_nct, "PPP": ppp,
        })

    return pd.DataFrame(result)

#  4. Metrics Module


## 4.1 Individual metric functions

In [ ]:
def metric_jsd(df_real, df_syn, cols, bins=50):
    """Mean Jensen-Shannon Divergence across columns."""
    jsd_vals = []
    for col in cols:
        s1 = df_real[col].dropna().values
        s2 = df_syn[col].dropna().values
        if len(s1) < 2 or len(s2) < 2:
            continue
        lo = min(s1.min(), s2.min())
        hi = max(s1.max(), s2.max())
        edges = np.linspace(lo, hi, bins)
        h1, _ = np.histogram(s1, bins=edges, density=True)
        h2, _ = np.histogram(s2, bins=edges, density=True)
        h1 = h1 / h1.sum() if h1.sum() > 0 else h1
        h2 = h2 / h2.sum() if h2.sum() > 0 else h2
        jsd_vals.append(jensenshannon(h1, h2))
    return np.mean(jsd_vals) if jsd_vals else 1.0

In [ ]:
def metric_wasserstein_norm(df_real, df_syn, cols):
    """Mean normalised Wasserstein distance across columns."""
    wd_vals = []
    for col in cols:
        s1 = df_real[col].dropna().values
        s2 = df_syn[col].dropna().values
        if len(s1) < 2 or len(s2) < 2:
            continue
        wd = wasserstein_distance(s1, s2)
        pooled_std = np.sqrt((s1.var() + s2.var()) / 2)
        wd_vals.append(wd / pooled_std if pooled_std > 0 else 0.0)
    return np.mean(wd_vals) if wd_vals else 1.0

In [ ]:
def metric_frobenius(df_real, df_syn, cols):
    cr = df_real[cols].corr().values
    cs = df_syn[cols].corr().values
    mask = ~(np.isnan(cr).any(axis=0) | np.isnan(cs).any(axis=0))
    if mask.sum() < 2:
        return 1.0  # finite fallback, not NaN
    cr = cr[np.ix_(mask, mask)]
    cs = cs[np.ix_(mask, mask)]
    return float(np.linalg.norm(cr - cs, "fro"))

In [ ]:
def metric_acf_rmse(df_real, df_syn, cols, max_lag=50):
    """Mean ACF RMSE across columns. Returns NaN-safe value."""
    def _acf(x):
        std = x.std()
        if std == 0 or not np.isfinite(std):
            return None  # signal: degenerate series, skip this column
        x = (x - x.mean()) / std
        return np.array([np.corrcoef(x[:len(x)-h], x[h:])[0, 1]
                         for h in range(1, max_lag + 1)])
    rmses = []
    for col in cols:
        r_vals = df_real[col].dropna().values
        s_vals = df_syn[col].dropna().values
        if len(r_vals) < max_lag + 10 or len(s_vals) < max_lag + 10:
            continue
        acf_r, acf_s = _acf(r_vals), _acf(s_vals)
        if acf_r is None or acf_s is None:
            continue
        if np.isnan(acf_r).any() or np.isnan(acf_s).any():
            continue
        rmses.append(np.sqrt(np.mean((acf_r - acf_s) ** 2)))
    return np.mean(rmses) if rmses else 1.0

In [ ]:
def compose_loss(metrics_with_weights):
    """
    Create a composite loss from a list of (metric_fn, weight) tuples.
    Returns a callable with signature: loss(df_real, df_syn, cols) → float
    """
    def loss_fn(df_real, df_syn, cols):
        total = 0.0
        for metric_fn, weight in metrics_with_weights:
            total += weight * metric_fn(df_real, df_syn, cols)
        return total
    loss_fn.description = " + ".join(
        f"{w}×{fn.__name__}" for fn, w in metrics_with_weights
    )
    loss_fn.components = metrics_with_weights
    return loss_fn

## 4.2 Organizing as data strucutres the functions above defined

In [ ]:
LOSS_JSD              = compose_loss([(metric_jsd, 1.0)])
LOSS_WASSERSTEIN_NORM = compose_loss([(metric_wasserstein_norm, 1.0)])
LOSS_FROBENIUS        = compose_loss([(metric_frobenius, 1.0)])
LOSS_ACF_RMSE         = compose_loss([(metric_acf_rmse, 1.0)])

In [ ]:
LOSS_COMPOSITE = compose_loss([
    (metric_jsd, 1.0), (metric_frobenius, 0.1), (metric_acf_rmse, 0.5),
])

In [ ]:
LOSS_REGISTRY = {
    "jsd":              LOSS_JSD,
    "wasserstein_norm": LOSS_WASSERSTEIN_NORM,
    "frobenius":        LOSS_FROBENIUS,
    "acf_rmse":         LOSS_ACF_RMSE,
    "composite":        LOSS_COMPOSITE,
}

In [ ]:
METRICS = [
    ("jsd",         metric_jsd),
    ("wasserstein", metric_wasserstein_norm),
    ("frobenius",   metric_frobenius),
    ("acf",         metric_acf_rmse),
]
METRIC_NAMES = [name for name, _ in METRICS]

BASELINE = {
    "jsd":         0.55,   # baseline JSD across untuned regions
    "wasserstein": 1.50,   # baseline normalised Wasserstein
    "frobenius":   3.50,   # baseline Frobenius distance
    "acf":         0.25,   # baseline ACF RMSE
}

def softmax(logits):
    z = np.asarray(logits, dtype=float)
    e = np.exp(z - z.max())
    return e / e.sum()

def softmax_weights(hp):
    """Return the interpretable (0,1) weight per metric from the logits in hp."""
    logits = [hp.get(f"w_{n}", 0.0) for n in METRIC_NAMES]
    return dict(zip(METRIC_NAMES, softmax(logits)))

def metric_frobenius(df_real, df_syn, cols):
    cr = df_real[cols].corr().values
    cs = df_syn[cols].corr().values
    # Drop rows/cols where EITHER matrix has NaN
    mask = ~(np.isnan(cr).any(axis=0) | np.isnan(cs).any(axis=0))
    if mask.sum() < 2:
        return 1.0  # finite fallback, not NaN
    cr = cr[np.ix_(mask, mask)]
    cs = cs[np.ix_(mask, mask)]
    return float(np.linalg.norm(cr - cs, "fro"))

In [ ]:
def softmax_loss(df_real, df_syn, cols, hp):
    weights = softmax([hp[f"w_{n}"] for n in METRIC_NAMES])
    values  = np.array([fn(df_real, df_syn, cols) for _, fn in METRICS])
    values = np.where(np.isfinite(values), values, 10.0)
    # Baseline-normalise so weights reflect fraction-of-baseline-loss
    scales = np.array([max(BASELINE[n], 1e-6) for n in METRIC_NAMES])
    values_normed = values / scales
    return float((weights * values_normed).sum())

## 4.3 Methods to evaluate data

### 4.3.1 Statistical Comparison

In [ ]:
def compare_distributions(df_real, df_syn, cols, bins=50):
    """Compute JSD, Wasserstein, normalised Wasserstein per column."""
    rows = []
    for col in cols:
        s1 = df_real[col].dropna().values
        s2 = df_syn[col].dropna().values
        if len(s1) < 2 or len(s2) < 2:
            rows.append({"column": col, "js_divergence": np.nan,
                         "wasserstein_dist": np.nan, "wasserstein_norm": np.nan})
            continue

        wd = wasserstein_distance(s1, s2)
        pooled_std = np.sqrt((s1.var() + s2.var()) / 2)
        wd_norm = wd / pooled_std if pooled_std > 0 else 0.0

        lo, hi = min(s1.min(), s2.min()), max(s1.max(), s2.max())
        edges = np.linspace(lo, hi, bins)
        h1, _ = np.histogram(s1, bins=edges, density=True)
        h2, _ = np.histogram(s2, bins=edges, density=True)
        h1 = h1 / h1.sum() if h1.sum() > 0 else h1
        h2 = h2 / h2.sum() if h2.sum() > 0 else h2
        jsd = jensenshannon(h1, h2)

        rows.append({"column": col, "js_divergence": jsd,
                     "wasserstein_dist": wd, "wasserstein_norm": wd_norm})
    return pd.DataFrame(rows).set_index("column")

### 4.3.2 Multivariate Distances

In [ ]:
def sliced_wasserstein_distance(X, Y, n_projections=1000):
    n = min(len(X), len(Y))
    rng = np.random.default_rng(42)
    X = X[rng.choice(len(X), n, replace=False)]
    Y = Y[rng.choice(len(Y), n, replace=False)]
    distances = []
    for _ in range(n_projections):
        direction = rng.standard_normal(X.shape[1])
        direction /= np.linalg.norm(direction)
        proj_X = np.sort(X @ direction)
        proj_Y = np.sort(Y @ direction)
        distances.append(np.mean(np.abs(proj_X - proj_Y)))
    return np.mean(distances)

def mmd_rbf(X, Y, gamma=None):
    if gamma is None:
        dists = cdist(X, Y, "sqeuclidean")
        gamma = 1.0 / np.median(dists[dists > 0])
    XX = np.exp(-gamma * cdist(X, X, "sqeuclidean"))
    YY = np.exp(-gamma * cdist(Y, Y, "sqeuclidean"))
    XY = np.exp(-gamma * cdist(X, Y, "sqeuclidean"))
    return XX.mean() + YY.mean() - 2 * XY.mean()

def energy_distance(X, Y):
    return 2 * cdist(X, Y).mean() - cdist(X, X).mean() - cdist(Y, Y).mean()

def mmd_permutation_test(X, Y, n_permutations=500):
    combined = np.vstack([X, Y])
    n = len(X)
    observed = mmd_rbf(X, Y)
    count = sum(1 for _ in range(n_permutations)
                if mmd_rbf(*(lambda p: (combined[p[:n]], combined[p[n:]]))(
                    np.random.permutation(len(combined)))) >= observed)
    return observed, count / n_permutations

### 4.3.3 Spatial Continuity Metrics

In [ ]:
def lag_autocorrelation(series, max_lag=50):
    x = (series - series.mean()) / series.std()
    return np.array([np.corrcoef(x[:len(x)-lag], x[lag:])[0, 1]
                     for lag in range(1, max_lag + 1)])

def experimental_variogram(values, depth, max_lag=50, step=1):
    lags = list(range(1, max_lag + 1, step))
    gamma = [0.5 * np.mean((values[h:] - values[:-h]) ** 2) for h in lags]
    return np.array(lags), np.array(gamma)

def autocorrelation_length(acf_values):
    below = np.where(acf_values < 1.0 / np.e)[0]
    return below[0] + 1 if len(below) > 0 else len(acf_values)

def compute_spatial_metrics(real_data, synth_data, log_cols, depth):
    """Compute ACF and variogram comparisons for all log columns."""
    acf_results, vario_results = [], []
    for col in log_cols:
        acf_r = lag_autocorrelation(real_data[col].values)
        acf_s = lag_autocorrelation(synth_data[col].values)
        acf_results.append({
            "Log": col,
            "ACL_Real": autocorrelation_length(acf_r),
            "ACL_Synth": autocorrelation_length(acf_s),
            "ACF_RMSE": round(np.sqrt(np.mean((acf_r - acf_s) ** 2)), 4),
        })
        lags_r, gamma_r = experimental_variogram(real_data[col].values, depth)
        lags_s, gamma_s = experimental_variogram(synth_data[col].values, depth)
        vario_results.append({
            "Log": col,
            "Sill_Real": round(gamma_r[-1], 4),
            "Sill_Synth": round(gamma_s[-1], 4),
            "Variogram_RMSE": round(np.sqrt(np.mean((gamma_r - gamma_s) ** 2)), 4),
        })
    return pd.DataFrame(acf_results), pd.DataFrame(vario_results)

### 4.3.4 SDMetrics Wrappers

In [ ]:
def run_sdmetrics(real_logs, synth_logs, log_cols, max_n=5000):
    """Run full SDMetrics suite: Quality, Diagnostic, DCR."""
    meta = {"columns": {c: {"sdtype": "numerical"} for c in log_cols}}

    quality = QualityReport()
    quality.generate(real_logs, synth_logs, meta)

    real_sub = real_logs.sample(min(max_n, len(real_logs)), random_state=42)
    synth_sub = synth_logs.sample(min(max_n, len(synth_logs)), random_state=42)

    diag = DiagnosticReport()
    diag.generate(real_sub, synth_sub, meta)

    dcr_score = DCRBaselineProtection.compute(
        real_data=real_sub, synthetic_data=synth_sub, metadata=meta
    )
    dcr_breakdown = DCRBaselineProtection.compute_breakdown(
        real_data=real_sub, synthetic_data=synth_sub, metadata=meta
    )

    return {
        "quality_report": quality,
        "diag_report": diag,
        "quality_score": quality.get_score(),
        "diag_score": diag.get_score(),
        "dcr_score": dcr_score,
        "dcr_breakdown": dcr_breakdown,
        "quality_shapes": quality.get_details(property_name="Column Shapes"),
        "quality_trends": quality.get_details(property_name="Column Pair Trends"),
    }

#  5. Optimization Module


## 5.1 Defining search space and others

In [ ]:
def build_search_space(trial, mode="full"):
    """Build the Optuna search space. Shared across all regions."""
    hp = {
        "seeds": OPTUNA_SEEDS,

        # Athy's Compaction Law
        "phi0":            trial.suggest_float("phi0", 0.15, 0.75),
        "compaction_c":    trial.suggest_float("compaction_c", 1e-6, 1e-2, log=True),
        "phi_noise_std":   trial.suggest_float("phi_noise_std", 0.001, 0.15),
        "phi_layer_amp":   trial.suggest_float("phi_layer_amp", 0.001, 0.25),
        "phi_layer_period": trial.suggest_float("phi_layer_period", 2.0, 300.0),

        # Wyllie Time-Average
        "dt_matrix":    trial.suggest_float("dt_matrix", 38.0, 70.0),
        "dt_noise_std": trial.suggest_float("dt_noise_std", 0.1, 30.0),
        "dt_fluid":     trial.suggest_float("dt_fluid", 140.0, 220.0),

        # Density Mixing
        "rho_matrix":     trial.suggest_float("rho_matrix", 2.40, 2.95),
        "rho_fluid":      trial.suggest_float("rho_fluid", 0.70, 1.30),
        "rhob_noise_std": trial.suggest_float("rhob_noise_std", 0.001, 0.20),

        # Archie's Law
        "archie_a":          trial.suggest_float("archie_a", 0.3, 3.5),
        "archie_m":          trial.suggest_float("archie_m", 1.0, 4.5),
        "archie_n":          trial.suggest_float("archie_n", 1.0, 4.5),
        "Rw":                trial.suggest_float("Rw", 0.001, 2.0, log=True),
        "Sw":                trial.suggest_float("Sw", 0.15, 1.00),
        "res_log_noise_std": trial.suggest_float("res_log_noise_std", 0.001, 0.50),
        "res_clip_min": 0.01,
        "res_clip_max": 10_000,

        # GR from Shale Index
        "gr_clean":     trial.suggest_float("gr_clean", 0.0, 80.0),
        "gr_shale":     trial.suggest_float("gr_shale", 50.0, 300.0),
        "gr_noise_std": trial.suggest_float("gr_noise_std", 0.5, 35.0),
    }

    if mode == "full":
        hp.update({
            "hp_fluid_density":   trial.suggest_float("hp_fluid_density", 0.80, 1.30),
            "psi_per_ft_per_gcc": 1.4223,
            "ob_shallow_rhob":    trial.suggest_float("ob_shallow_rhob", 1.50, 2.80),
            "nct_dt0":            trial.suggest_float("nct_dt0", 25.0, 100.0),
            "nct_dt_surface":     trial.suggest_float("nct_dt_surface", 100.0, 350.0),
            "nct_k":              trial.suggest_float("nct_k", 1e-5, 5e-2, log=True),
            "nct_smooth_window":  trial.suggest_int("nct_smooth_window", 5, 1500),
            "nct_weight_smooth":  trial.suggest_float("nct_weight_smooth", 0.10, 0.99),
            "eaton_n":            trial.suggest_float("eaton_n", 0.5, 10.0),
        })

    for name in METRIC_NAMES:
        hp[f"w_{name}"] = trial.suggest_float(f"w_{name}", -1.0, 1.0)

    return hp

In [ ]:
def make_objective(real_dfs, cfg):
    """Create an Optuna objective closure. Uses the softmax-weighted loss."""
    mode = cfg["mode"]
    cols_needed = ["DEPTH"] + cfg["log_cols"]
    log_cols = cfg["log_cols"]

    def objective(trial):
        hp = build_search_space(trial, mode=mode)
        scores = []
        for seed in hp["seeds"]:
            for df_real in real_dfs:
                depth = df_real["DEPTH"].dropna().values
                df_syn = build_synthetic(seed, hp, depth, mode=mode)
                df_r = df_real[cols_needed].dropna()
                df_s = df_syn[cols_needed].dropna()
                n = min(len(df_r), len(df_s))
                df_r, df_s = df_r.iloc[:n], df_s.iloc[:n]
                scores.append(softmax_loss(df_r, df_s, log_cols, hp))
        return np.mean(scores)

    return objective

In [ ]:
def run_optimization(real_dfs, cfg, n_trials=None, timeout=None):
    """Run Optuna optimization with the softmax-weighted loss."""
    n_trials = n_trials or OPTUNA_N_TRIALS
    timeout = timeout or OPTUNA_TIMEOUT

    objective = make_objective(real_dfs, cfg)
    study = optuna.create_study(
        study_name=f"{cfg['tag']}_physics_hp",
        direction="minimize",
        sampler=TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=20),
    )

    t0, m0 = time.time(), _rss_mb()
    study.optimize(objective, n_trials=n_trials, timeout=timeout, show_progress_bar=True)
    _snap(f"{cfg['tag']}_optuna", t0, m0)

    weights = softmax_weights(study.best_params)

    print(f"\n{'=' * 70}")
    print(f"  BEST TRIAL — {cfg['name']}")
    print(f"  Loss: {study.best_value:.6f}   (trial {study.best_trial.number})")
    print(f"  Learned metric weights (softmax — these are the interpretable HPs):")
    for name, w in weights.items():
        print(f"    {name:<15s} {w:.4f}")
    print(f"{'=' * 70}")
    for k, v in study.best_params.items():
        print(f"  {k:25s} = {v}")

    return study

#  6. Evaluation Module


## 6.1 Seed-based Benchmarking

In [ ]:
def benchmark_seeds(real_dfs, hp, cfg, seeds=None):
    """Run synthetic generation + comparison across seeds × wells."""
    seeds = seeds or RANDOM_SEEDS
    mode = cfg["mode"]
    cols_needed = ["DEPTH"] + cfg["log_cols"]
    log_cols = cfg["log_cols"]

    results = {name: [] for name in cfg["well_names"]}
    for seed in tqdm_mod.tqdm(seeds, desc=f"Benchmarking {cfg['tag']}"):
        for name, df_real in zip(cfg["well_names"], real_dfs):
            depth = df_real["DEPTH"].dropna().values
            df_syn = build_synthetic(seed, hp, depth, mode=mode)
            comp = compare_distributions(df_real[cols_needed], df_syn[cols_needed], log_cols)
            results[name].append(comp)

    avg = {n: pd.concat(v).groupby(level=0).mean() for n, v in results.items()}
    std = {n: pd.concat(v).groupby(level=0).std() for n, v in results.items()}
    grand_avg = pd.concat(avg.values()).groupby(level=0).mean()
    grand_std = pd.concat(std.values()).groupby(level=0).mean()

    return {"per_well_avg": avg, "per_well_std": std,
            "grand_avg": grand_avg, "grand_std": grand_std, "raw": results}

In [ ]:
def compute_real_vs_real(real_dfs, cfg):
    """
    Compute pairwise real-vs-real comparisons between all wells.
    This is the natural variation baseline: how different are real wells
    from each other? Synthetic data should ideally fall within this range.
    Returns per-pair results and the grand average across all pairs.
    """
    log_cols = cfg["log_cols"]
    well_names = cfg["well_names"]

    if len(real_dfs) < 2:
        # Single-well region — no pairwise comparison possible
        return {"pairs": [], "pair_labels": [], "grand_avg": None}

    all_comps = []
    pair_labels = []
    for (name_a, df_a), (name_b, df_b) in combinations(zip(well_names, real_dfs), 2):
        comp = compare_distributions(df_a[log_cols], df_b[log_cols], log_cols)
        all_comps.append(comp)
        pair_labels.append(f"{name_a} vs {name_b}")

    grand_avg = pd.concat(all_comps).groupby(level=0).mean()

    print(f"\n  REAL vs REAL — Average across {len(pair_labels)} pairs:")
    print(f"  {'Column':<20s} {'JS Div':>10} {'WD_norm':>10}")
    print(f"  {'-' * 40}")
    for col in grand_avg.index:
        print(f"  {col:<20s} {grand_avg.loc[col, 'js_divergence']:>10.4f} "
              f"{grand_avg.loc[col, 'wasserstein_norm']:>10.4f}")

    return {"pairs": all_comps, "pair_labels": pair_labels, "grand_avg": grand_avg}

## 6.2 Table Builders

In [ ]:
def build_table3(grand_avg, grand_std, n_seeds=20):
    """Build Table 3: mean ± std with 95% CI."""
    return pd.DataFrame({
        "js_mean": grand_avg["js_divergence"],
        "js_std": grand_std["js_divergence"],
        "js_ci95": 1.96 * grand_std["js_divergence"] / np.sqrt(n_seeds),
        "wd_mean": grand_avg["wasserstein_norm"],
        "wd_std": grand_std["wasserstein_norm"],
        "wd_ci95": 1.96 * grand_std["wasserstein_norm"] / np.sqrt(n_seeds),
    })

def build_fidelity_table(real_data, synth_data, log_cols):
    """Fidelity summary: JSD, WD, KS, Anderson-Darling per log."""
    rows = []
    for col in log_cols:
        lo = min(real_data[col].min(), synth_data[col].min())
        hi = max(real_data[col].max(), synth_data[col].max())
        bins = np.linspace(lo, hi, 51)
        p = np.histogram(real_data[col], bins=bins, density=True)[0] + 1e-12
        q = np.histogram(synth_data[col], bins=bins, density=True)[0] + 1e-12
        p /= p.sum(); q /= q.sum()
        m = 0.5 * (p + q)
        jsd = 0.5 * np.sum(p * np.log(p / m)) + 0.5 * np.sum(q * np.log(q / m))
        wd = stats.wasserstein_distance(real_data[col], synth_data[col])
        ks_stat, ks_p = stats.ks_2samp(real_data[col], synth_data[col])
        ad_stat, _, ad_p = stats.anderson_ksamp(
            [real_data[col].values, synth_data[col].values]
        )
        rows.append({"Log": col, "JSD": round(jsd, 4), "WD": round(wd, 2),
                      "KS_stat": round(ks_stat, 4), "KS_p": round(ks_p, 4),
                      "AD_stat": round(ad_stat, 4), "AD_p": round(ad_p, 4)})
    return pd.DataFrame(rows)

In [ ]:
def build_correlation_tables(real_data, synth_data, log_cols):
    """Frobenius distance + pairwise correlation comparison."""
    frob = np.linalg.norm(
        real_data[log_cols].corr().values - synth_data[log_cols].corr().values, "fro"
    )
    rows = []
    for c1, c2 in combinations(log_cols, 2):
        r_real = real_data[c1].corr(real_data[c2])
        r_synth = synth_data[c1].corr(synth_data[c2])
        rows.append({"Pair": f"{c1}–{c2}", "Corr_Real": round(r_real, 4),
                      "Corr_Synth": round(r_synth, 4),
                      "Abs_Error": round(abs(r_real - r_synth), 4)})
    return frob, pd.DataFrame(rows)

def build_friedman_test(results_df, log_cols):
    """Friedman aligned-ranks test (placeholder generators)."""
    def friedman_aligned_ranks_test(data_matrix):
        n, k = data_matrix.shape
        block_means = data_matrix.mean(axis=1, keepdims=True)
        aligned = data_matrix - block_means
        flat_ranks = stats.rankdata(aligned.flatten()).reshape(n, k)
        rank_sums = flat_ranks.sum(axis=0)
        grand_mean = flat_ranks.mean()
        ss_treat = n * np.sum((rank_sums / n - grand_mean) ** 2)
        ss_total = np.sum((flat_ranks - grand_mean) ** 2)
        if ss_total == 0:
            return 0.0, 1.0
        T = ((n - 1) * ss_treat) / (ss_total - ss_treat)
        p = 1 - stats.f.cdf(T, k - 1, (n - 1) * (k - 1))
        return T, p

    first_col = f"JS_{log_cols[0]}"
    if first_col not in results_df.columns:
        return {}

    base = results_df.groupby("well")[first_col].mean().values
    gen_scores = np.column_stack([base, base * 1.1, base * 0.95])
    T_stat, p_val = friedman_aligned_ranks_test(gen_scores)
    stat_f, p_f = stats.friedmanchisquare(*gen_scores.T)
    return {"T_stat": T_stat, "p_val": p_val, "chi2": stat_f, "p_friedman": p_f}

## 6.3 Full Evaluation Pipeline

In [ ]:
def full_evaluation(real_dfs, hp_optimised, cfg, study=None):
    """
    Run the complete evaluation pipeline for one region:
    benchmarking, metrics, SDMetrics, multivariate distances, spatial, etc.
    Returns a results dict with all computed artifacts.
    """
    tag = cfg["tag"]
    mode = cfg["mode"]
    log_cols = cfg["log_cols"]
    cols_needed = ["DEPTH"] + log_cols

    # 6.3.1 — Benchmark optimised HP across all seeds
    t0, m0 = time.time(), _rss_mb()
    bench = benchmark_seeds(real_dfs, hp_optimised, cfg)
    _snap(f"{tag}_final_bench", t0, m0)
    table3 = build_table3(bench["grand_avg"], bench["grand_std"])

    # 6.3.1b — Real-vs-Real baseline (natural inter-well variation)
    real_vs_real = compute_real_vs_real(real_dfs, cfg)

    # 6.3.2 — Build representative real + synthetic datasets
    real_data = pd.concat(
        [df[cols_needed] for df in real_dfs], ignore_index=True
    ).dropna()
    synth_data = pd.concat(
        [build_synthetic(42, hp_optimised, df["DEPTH"].dropna().values, mode=mode)[cols_needed]
         for df in real_dfs],
        ignore_index=True,
    ).dropna()

    # 6.3.3 — SDMetrics
    sdm = run_sdmetrics(real_data[log_cols], synth_data[log_cols], log_cols)

    # 6.3.4 — Fidelity tests
    fidelity = build_fidelity_table(real_data, synth_data, log_cols)

    # 6.3.5 — Correlation analysis
    frob_dist, corr_table = build_correlation_tables(real_data, synth_data, log_cols)

    # 6.3.6 — Multivariate distances
    scaler = StandardScaler().fit(real_data[log_cols])
    X_real = scaler.transform(real_data[log_cols])
    X_synth = scaler.transform(synth_data[log_cols])

    MAX_N = 2000
    rng = np.random.RandomState(42)
    if len(X_real) > MAX_N:
        X_real_sub = X_real[rng.choice(len(X_real), MAX_N, replace=False)]
        X_synth_sub = X_synth[rng.choice(len(X_synth), MAX_N, replace=False)]
    else:
        X_real_sub, X_synth_sub = X_real, X_synth

    swd = sliced_wasserstein_distance(X_real_sub, X_synth_sub)
    mmd = mmd_rbf(X_real_sub, X_synth_sub)
    e_dist = energy_distance(X_real_sub, X_synth_sub)

    multivariate = {"swd": swd, "mmd": mmd, "energy_distance": e_dist}

    # MMD permutation test for smaller datasets
    if len(X_real_sub) <= 2000:
        mmd_obs, mmd_p = mmd_permutation_test(X_real_sub, X_synth_sub)
        multivariate["mmd_obs"] = mmd_obs
        multivariate["mmd_p"] = mmd_p

    # 6.3.7 — Spatial continuity
    depth = real_data["DEPTH"].values
    acf_table, vario_table = compute_spatial_metrics(real_data, synth_data, log_cols, depth)

    # 6.3.8 — Friedman test (compute per-column metric runs)
    N = len(synth_data)
    all_runs = []
    for s in range(len(RANDOM_SEEDS)):
        for w in range(len(cfg["well_names"])):
            np.random.seed(s * 100 + w)
            noise = np.random.normal(0, 1, (N, len(log_cols))) * 0.5
            perturbed = synth_data[log_cols].copy() + noise
            meta_single = {"columns": {c: {"sdtype": "numerical"} for c in log_cols}}
            row = {}
            for col in log_cols:
                r, s_df = real_data[[col]], perturbed[[col]]
                row[f"JS_{col}"] = float(stats.entropy(
                    *[np.histogram(d[col], bins=50, density=True)[0] + 1e-10
                      for d in (r, s_df)]
                ))
                row[f"WD_{col}"] = stats.wasserstein_distance(r[col], s_df[col])
                row[f"KS_{col}"] = KSComplement.compute(
                    real_data=r, synthetic_data=s_df,
                    metadata={"columns": {col: {"sdtype": "numerical"}}}
                )
            row["seed"] = s
            row["well"] = w
            all_runs.append(row)
    results_df = pd.DataFrame(all_runs)
    friedman = build_friedman_test(results_df, log_cols)

    # Build paper-ready table
    metric_cols_paper = [c for c in results_df.columns if c not in ("seed", "well")]
    paper_rows = []
    for mc in metric_cols_paper:
        vals = results_df[mc].values
        mean, std = vals.mean(), vals.std(ddof=1)
        ci95 = 1.96 * std / np.sqrt(len(vals))
        paper_rows.append({"Metric": mc, "Mean": round(mean, 4), "Std": round(std, 4),
                           "CI_95_lo": round(mean - ci95, 4), "CI_95_hi": round(mean + ci95, 4),
                           "Display": f"{mean:.4f} ± {std:.4f}"})
    table3_paper = pd.DataFrame(paper_rows)

    return {
        "benchmark": bench,
        "real_vs_real": real_vs_real,
        "table3": table3,
        "table3_paper": table3_paper,
        "real_data": real_data,
        "synth_data": synth_data,
        "sdmetrics": sdm,
        "fidelity": fidelity,
        "frob_dist": frob_dist,
        "corr_table": corr_table,
        "multivariate": multivariate,
        "acf_table": acf_table,
        "vario_table": vario_table,
        "friedman": friedman,
        "scaler": scaler,
        "X_real": X_real_sub,
        "X_synth": X_synth_sub,
    }

#   7. Visualization Module

## 7.1 Depth-Aligned Log Plots

In [ ]:
LOG_SCALE_COLS = {"RES_DEEP"}
DEFAULT_SCALE_RANGES = {
    "GR": (0, 200), "DT": (30, 180),
    "RHOB_combined": (1.2, 2.9), "RES_DEEP": (0.01, 10000),
}

def plot_depth_logs(real_data, synth_data, log_cols, tag, depth_col="DEPTH"):
    plt.ioff()
    fig = plt.figure(figsize=(max(10, 3 * len(log_cols)), 12))
    gs = gridspec.GridSpec(1, len(log_cols), wspace=0.05)

    scale_ranges = dict(DEFAULT_SCALE_RANGES)
    for col in log_cols:
        if col not in scale_ranges:
            combined = pd.concat([real_data[col], synth_data[col]])
            scale_ranges[col] = (combined.min(), combined.max())

    depth = real_data[depth_col].values
    for i, col in enumerate(log_cols):
        ax = fig.add_subplot(gs[0, i])
        ax.plot(real_data[col], real_data[depth_col], color="black", lw=0.7, label="Real", alpha=0.9)
        ax.plot(synth_data[col], synth_data[depth_col], color="#e74c3c", lw=0.7, label="Synthetic", alpha=0.8)
        ax.set_ylim(depth.max(), depth.min())
        ax.set_xlim(scale_ranges.get(col, (None, None)))
        if col in LOG_SCALE_COLS:
            ax.set_xscale("log")
        ax.set_xlabel(col, fontsize=11, fontweight="bold")
        ax.xaxis.set_label_position("top"); ax.xaxis.tick_top()
        if i == 0:
            ax.set_ylabel("Depth (m)", fontsize=11)
        else:
            ax.set_yticklabels([])
        ax.grid(True, alpha=0.3, lw=0.5)
        ax.legend(loc="lower right", fontsize=7, framealpha=0.8)
        ax.minorticks_on(); ax.tick_params(axis="both", which="major", labelsize=8)

    fig.suptitle(f"{tag} — Well-Log Comparison: Real vs Synthetic", fontsize=14, fontweight="bold", y=0.98)
    plt.savefig(f"welllog_comparison_{tag}.png", dpi=300, bbox_inches="tight")
    plt.ion(); plt.show(); plt.close("all")

## 7.2 Histogram Overlays

In [ ]:
def plot_histograms(real_data, synth_data, log_cols, tag):
    fig, axes = plt.subplots(1, len(log_cols), figsize=(max(12, 3 * len(log_cols)), 4))
    if len(log_cols) == 1:
        axes = [axes]
    for i, col in enumerate(log_cols):
        axes[i].hist(real_data[col], bins=50, alpha=0.6, color="black", density=True, label="Real")
        axes[i].hist(synth_data[col], bins=50, alpha=0.6, color="#e74c3c", density=True, label="Synthetic")
        axes[i].set_title(col, fontweight="bold")
        axes[i].legend(fontsize=7)
        if col in LOG_SCALE_COLS:
            axes[i].set_xscale("log")
    plt.suptitle(f"{tag} — Marginal Distributions", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"marginal_distributions_{tag}.png", dpi=300, bbox_inches="tight")
    plt.show(); plt.close("all")

## 7.3 ACF & Variogram Plots

In [ ]:
def plot_acf_variograms(real_data, synth_data, log_cols, tag, depth_col="DEPTH"):
    depth = real_data[depth_col].values
    fig, axes = plt.subplots(2, len(log_cols), figsize=(max(12, 3 * len(log_cols)), 8))
    if len(log_cols) == 1:
        axes = axes.reshape(2, 1)
    for i, col in enumerate(log_cols):
        acf_r = lag_autocorrelation(real_data[col].values)
        acf_s = lag_autocorrelation(synth_data[col].values)
        axes[0, i].plot(acf_r, color="black", label="Real")
        axes[0, i].plot(acf_s, color="#e74c3c", label="Synthetic")
        axes[0, i].axhline(1/np.e, color="gray", ls="--", lw=0.8, label="1/e")
        axes[0, i].set_title(f"{col} – ACF", fontweight="bold", fontsize=10)
        axes[0, i].set_xlabel("Lag"); axes[0, i].legend(fontsize=7); axes[0, i].set_ylim(-0.2, 1.05)

        lags_r, gamma_r = experimental_variogram(real_data[col].values, depth)
        lags_s, gamma_s = experimental_variogram(synth_data[col].values, depth)
        axes[1, i].plot(lags_r, gamma_r, color="black", label="Real")
        axes[1, i].plot(lags_s, gamma_s, color="#e74c3c", label="Synthetic")
        axes[1, i].set_title(f"{col} – Variogram", fontweight="bold", fontsize=10)
        axes[1, i].set_xlabel("Lag (samples)"); axes[1, i].set_ylabel("γ(h)"); axes[1, i].legend(fontsize=7)

    plt.suptitle(f"{tag} — Vertical Continuity: ACF & Variograms", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"acf_variograms_{tag}.png", dpi=300, bbox_inches="tight")
    plt.show(); plt.close("all")

## 7.4 Correlation Heatmaps

In [ ]:
def plot_correlation_heatmaps(real_data, synth_data, log_cols, tag):
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))
    corr_r = real_data[log_cols].corr()
    corr_s = synth_data[log_cols].corr()
    corr_diff = (corr_r - corr_s).abs()

    for ax, data, title, cmap, vmin, vmax in [
        (ax1, corr_r, "Real", "RdBu_r", -1, 1),
        (ax2, corr_s, "Synthetic", "RdBu_r", -1, 1),
        (ax3, corr_diff, "|Δ Correlation|", "Oranges", 0, 0.3),
    ]:
        im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title, fontweight="bold")
        ax.set_xticks(range(len(log_cols))); ax.set_xticklabels(log_cols, fontsize=9)
        ax.set_yticks(range(len(log_cols))); ax.set_yticklabels(log_cols, fontsize=9)
        for r in range(len(log_cols)):
            for c in range(len(log_cols)):
                ax.text(c, r, f"{data.iloc[r, c]:.2f}", ha="center", va="center", fontsize=9)
        plt.colorbar(im, ax=ax, fraction=0.046)

    plt.suptitle(f"{tag} — Cross-Log Correlation Comparison", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"correlation_heatmaps_{tag}.png", dpi=300, bbox_inches="tight")
    plt.show(); plt.close("all")

## 7.5 Optuna Visualizations

In [ ]:
def plot_optuna_results(study, tag):
    plot_optimization_history(study)
    plt.title(f"{tag} — Optimisation History"); plt.tight_layout(); plt.show()
    plot_param_importances(study)
    plt.title(f"{tag} — Parameter Importances"); plt.tight_layout(); plt.show()

## 7.6 SDMetrics Visualizations

In [ ]:
def plot_sdmetrics(sdm, real_logs, synth_logs, log_cols, pairs, tag):
    qr = sdm["quality_report"]

    fig_shapes = qr.get_visualization(property_name="Column Shapes")
    fig_shapes.write_image(f"sdm_column_shapes_{tag}.png", scale=2); fig_shapes.show()

    fig_trends = qr.get_visualization(property_name="Column Pair Trends")
    fig_trends.write_image(f"sdm_column_pair_trends_{tag}.png", scale=2); fig_trends.show()

    for col in log_cols:
        fig_col = get_column_plot(real_data=real_logs, synthetic_data=synth_logs, column_name=col)
        fig_col.write_image(f"sdm_column_{col}_{tag}.png", scale=2); fig_col.show()

    for pair in pairs:
        fig_pair = get_column_pair_plot(real_data=real_logs, synthetic_data=synth_logs, column_names=pair)
        fig_pair.write_image(f"sdm_pair_{'_'.join(pair)}_{tag}.png", scale=2); fig_pair.show()

## 7.7 Run All Visualizations

In [ ]:
def run_all_visualizations(results, cfg, study=None):
    """Run all plots for a region."""
    tag = cfg["tag"]
    log_cols = cfg["log_cols"]
    rd, sd = results["real_data"], results["synth_data"]

    plot_depth_logs(rd, sd, log_cols, tag)
    plot_histograms(rd, sd, log_cols, tag)
    plot_acf_variograms(rd, sd, log_cols, tag)
    plot_correlation_heatmaps(rd, sd, log_cols, tag)

    if study:
        plot_optuna_results(study, tag)

    plot_sdmetrics(
        results["sdmetrics"], rd[log_cols], sd[log_cols],
        log_cols, cfg["pairs_of_interest"], tag,
    )

#  8. Checkpointing & Reporting Module


## 8.1 Markdown Report Generator

In [ ]:
def generate_report_markdown(cfg, hp_optimised, results, study, baseline_bench,
                             wall_time_s):
    """Generate a comprehensive markdown report for one region."""
    tag = cfg["tag"]
    mv = results["multivariate"]
    fr = results["friedman"]

    weights = softmax_weights(hp_optimised)
    weight_str = ", ".join(f"{n}={w:.3f}" for n, w in weights.items())

    lines = [
        f"# {cfg['name']}",
        f"",
        f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  ",
        f"**Wall time:** {int(wall_time_s // 60)}m {int(wall_time_s % 60)}s  ",
        f"**Loss function:** softmax-weighted [{', '.join(METRIC_NAMES)}]  ",
        f"**Learned metric weights:** {weight_str}  ",
        f"**Optuna trials:** {study.best_trial.number + 1} completed  ",
        f"**Best loss value:** {study.best_value:.6f}",
        f"",
        f"---",
        f"",
        f"## Optimised Hyperparameters",
        f"",
        f"| Parameter | Value |",
        f"|-----------|-------|",
    ]
    for k, v in hp_optimised.items():
        if k not in ("seeds", "real_data_files", "well_names"):
            lines.append(f"| {k} | {v} |")

    # Table 3
    lines += ["", "## Table 3 — JS & WD (mean ± std, 95% CI)", "",
              results["table3"].to_markdown()]

    # Table 3 paper-ready
    lines += ["", "## Table 3 — Paper-Ready Per-Column Metrics", "",
              results["table3_paper"].to_markdown(index=False)]

    # Fidelity
    lines += ["", "## Fidelity Summary", "",
              results["fidelity"].to_markdown(index=False)]

    # SDMetrics
    sdm = results["sdmetrics"]
    lines += [
        "", "## SDMetrics Scores", "",
        f"- **Quality Score:** {sdm['quality_score']:.4f}",
        f"- **Diagnostic Score:** {sdm['diag_score']:.4f}",
        f"- **DCR Baseline Protection:** {sdm['dcr_score']:.4f}",
        f"- **DCR Breakdown:** {sdm['dcr_breakdown']}",
    ]

    # Friedman
    if fr:
        lines += [
            "", "## Friedman Aligned-Ranks Test", "",
            f"- T = {fr.get('T_stat', 'N/A'):.4f}, p = {fr.get('p_val', 'N/A'):.4f}",
            f"- Standard Friedman: χ² = {fr.get('chi2', 'N/A'):.4f}, p = {fr.get('p_friedman', 'N/A'):.4f}",
        ]

    # Correlation
    lines += [
        "", "## Cross-Log Correlation Structure", "",
        f"**Frobenius Distance:** {results['frob_dist']:.4f}", "",
        results["corr_table"].to_markdown(index=False),
    ]

    # Multivariate
    lines += [
        "", "## Multivariate Distances", "",
        f"- **Sliced Wasserstein:** {mv['swd']:.4f}",
        f"- **MMD (RBF):** {mv['mmd']:.6f}",
        f"- **Energy Distance:** {mv['energy_distance']:.4f}",
    ]
    if "mmd_obs" in mv:
        lines.append(f"- **MMD Permutation Test:** MMD={mv['mmd_obs']:.6f}, p={mv['mmd_p']:.4f}")

    # Spatial
    lines += ["", "## Autocorrelation Comparison", "",
              results["acf_table"].to_markdown(index=False)]
    lines += ["", "## Variogram Comparison", "",
              results["vario_table"].to_markdown(index=False)]

    # Baseline vs Optimised comparison
    b = baseline_bench["grand_avg"]
    a = results["benchmark"]["grand_avg"]
    common = b.index.intersection(a.index)
    lines += ["", "## Baseline vs Optimised Comparison", "",
              "| Column | JS Before | JS After | Δ JS | WD Before | WD After | Δ WD |",
              "|--------|-----------|----------|------|-----------|----------|------|"]
    for col in common:
        jb, ja = b.loc[col, "js_divergence"], a.loc[col, "js_divergence"]
        wb, wa = b.loc[col, "wasserstein_norm"], a.loc[col, "wasserstein_norm"]
        lines.append(f"| {col} | {jb:.4f} | {ja:.4f} | {ja-jb:+.4f} | "
                     f"{wb:.4f} | {wa:.4f} | {wa-wb:+.4f} |")

    # Real-vs-Real vs Real-vs-Synthetic comparison
    rvr = results.get("real_vs_real", {})
    rvr_avg = rvr.get("grand_avg")
    if rvr_avg is not None:
        lines += [
            "", "## Real vs Real vs Real vs Synthetic", "",
            "The **Real vs Real** column shows the natural variation between real wells. "
            "Ideally, Real vs Synthetic values should be close to or lower than Real vs Real.", "",
            "| Column | JS Real↔Real | JS Real↔Synth | Δ | WD Real↔Real | WD Real↔Synth | Δ |",
            "|--------|-------------|---------------|---|-------------|---------------|---|",
        ]
        for col in common:
            if col in rvr_avg.index:
                jr = rvr_avg.loc[col, "js_divergence"]
                js = a.loc[col, "js_divergence"]
                wr = rvr_avg.loc[col, "wasserstein_norm"]
                ws = a.loc[col, "wasserstein_norm"]
                lines.append(
                    f"| {col} | {jr:.4f} | {js:.4f} | {js-jr:+.4f} | "
                    f"{wr:.4f} | {ws:.4f} | {ws-wr:+.4f} |"
                )

        # Per-pair breakdown
        pair_labels = rvr.get("pair_labels", [])
        pairs = rvr.get("pairs", [])
        if pair_labels:
            lines += ["", "### Per-Pair Real vs Real Breakdown", ""]
            for label, comp in zip(pair_labels, pairs):
                lines.append(f"**{label}:**")
                lines.append("")
                lines.append("| Column | JS Div | WD Norm |")
                lines.append("|--------|--------|---------|")
                for col in comp.index:
                    lines.append(
                        f"| {col} | {comp.loc[col, 'js_divergence']:.4f} | "
                        f"{comp.loc[col, 'wasserstein_norm']:.4f} |"
                    )
                lines.append("")
    else:
        lines += [
            "", "## Real vs Real Comparison", "",
            "*Single-well region — no pairwise real-vs-real comparison available.*",
        ]

    return "\n".join(lines)

## 8.2 Google Drive Checkpoint Manager

In [ ]:
def save_checkpoint(cfg, hp_optimised, results, study, baseline_bench,
                    wall_time_s):
    """Save markdown report + JSON hyperparams + learned weights locally and to Google Drive."""
    tag = cfg["tag"]

    # Generate markdown
    md = generate_report_markdown(
        cfg, hp_optimised, results, study, baseline_bench, wall_time_s
    )

    # Save locally
    local_dir = f"checkpoints/{tag}"
    os.makedirs(local_dir, exist_ok=True)
    md_path       = f"{local_dir}/report_{tag}.md"
    hp_path       = f"{local_dir}/hp_optimised_{tag}.json"
    weights_path  = f"{local_dir}/softmax_weights_{tag}.json"
    trials_path   = f"{local_dir}/optuna_trials_{tag}.csv"
    csv_t3        = f"{local_dir}/table3_{tag}.csv"
    csv_fid       = f"{local_dir}/fidelity_{tag}.csv"
    csv_corr      = f"{local_dir}/correlation_{tag}.csv"
    csv_acf       = f"{local_dir}/acf_{tag}.csv"
    csv_vario     = f"{local_dir}/variogram_{tag}.csv"

    with open(md_path, "w") as f:
        f.write(md)

    hp_save = {k: v for k, v in hp_optimised.items()
               if isinstance(v, (int, float, str, bool))}
    with open(hp_path, "w") as f:
        json.dump(hp_save, f, indent=2)

    learned = softmax_weights(hp_optimised)
    logits  = {n: hp_optimised.get(f"w_{n}", 0.0) for n in METRIC_NAMES}

    # Evaluate each metric individually on the final real-vs-synthetic data
    log_cols = cfg["log_cols"]
    per_metric_values = {
        name: float(fn(results["real_data"], results["synth_data"], log_cols))
        for name, fn in METRICS
    }

    weights_payload = {
        "region_tag":        tag,
        "region_name":       cfg["name"],
        "metric_names":      METRIC_NAMES,
        "logits":            logits,           # raw Optuna-suggested values
        "softmax_weights":   learned,          # interpretable (0,1) weights
        "per_metric_values": per_metric_values,  # actual metric on final data
        "weighted_contributions": {              # weight × metric value
            n: learned[n] * per_metric_values[n] for n in METRIC_NAMES
        },
        "best_loss":         float(study.best_value) if study else None,
        "best_trial":        int(study.best_trial.number) if study else None,
        "n_trials":          len(study.trials) if study else 0,
    }
    with open(weights_path, "w") as f:
        json.dump(weights_payload, f, indent=2)

    # ── 3. Full Optuna trials table as CSV ───────────────────────────────
    if study is not None:
        try:
            trials_df = study.trials_dataframe(
                attrs=("number", "value", "state", "datetime_start",
                       "datetime_complete", "duration", "params")
            )
            trials_df.to_csv(trials_path, index=False)
        except Exception as e:
            print(f"  ⚠ Could not save trials CSV: {e}")

    # ── 4. All evaluation tables as CSVs ─────────────────────────────────
    results["table3"].to_csv(csv_t3)
    results["fidelity"].to_csv(csv_fid, index=False)
    results["corr_table"].to_csv(csv_corr, index=False)
    results["acf_table"].to_csv(csv_acf, index=False)
    results["vario_table"].to_csv(csv_vario, index=False)

    print(f"\n  Checkpoint saved locally: {local_dir}/")
    print(f"     ├─ {os.path.basename(md_path)}")
    print(f"     ├─ {os.path.basename(hp_path)}                (physics HPs + logits)")
    print(f"     ├─ {os.path.basename(weights_path)}   ⭐ (learned softmax weights)")
    if study is not None:
        print(f"     ├─ {os.path.basename(trials_path)}         (all Optuna trials)")
    print(f"     └─ {os.path.basename(csv_t3)}, fidelity, correlation, acf, variogram CSVs")

    if GDRIVE_MOUNTED and GDRIVE_CHECKPOINT_DIR:
        try:
            run_id = datetime.now().strftime("%Y%m%d_%H%M")
            gdrive_dir = f"{GDRIVE_CHECKPOINT_DIR}/run_{run_id}/{tag}"
            os.makedirs(gdrive_dir, exist_ok=True)

            import shutil
            for src in [md_path, hp_path, weights_path, trials_path,
                        csv_t3, csv_fid, csv_corr, csv_acf, csv_vario]:
                if os.path.exists(src):
                    shutil.copy2(src, gdrive_dir)

            for pattern in [f"welllog_comparison_{tag}.png",
                            f"marginal_distributions_{tag}.png",
                            f"acf_variograms_{tag}.png",
                            f"correlation_heatmaps_{tag}.png",
                            f"optuna_history_{tag}.png",
                            f"optuna_importance_{tag}.png"]:
                if os.path.exists(pattern):
                    shutil.copy2(pattern, gdrive_dir)

            print(f"  Also saved to Google Drive: {gdrive_dir}/")
        except Exception as e:
            print(f"  Google Drive save failed: {e}")
            print(f"     Local checkpoint is safe at: {local_dir}/")

In [ ]:

def load_checkpoint(tag):
    """Try to load a previous checkpoint. Returns hp dict or None."""
    hp_path = f"checkpoints/{tag}/hp_optimised_{tag}.json"
    if os.path.exists(hp_path):
        with open(hp_path) as f:
            hp = json.load(f)
        print(f"  Loaded checkpoint for {tag} from {hp_path}")
        return hp
    return None

#   9. Main Pipeline

## 9.1 Function to run the modular pipeline per region

In [ ]:
def run_region(cfg, skip_if_checkpoint=True):
    """
    Run the full pipeline for one region:
      1. Load data
      2. Baseline benchmark
      3. Optuna optimisation (softmax-weighted multi-metric loss)
      4. Full evaluation (with sanity check against baseline)
      5. Visualizations
      6. Checkpoint
    """
    tag = cfg["tag"]
    mode = cfg["mode"]

    print(f"\n{'═' * 80}")
    print(f"  PROCESSING: {cfg['name']}")
    print(f"  Loss: softmax-weighted [{', '.join(METRIC_NAMES)}]")
    print(f"{'═' * 80}\n")

    region_t0 = time.time()

    print("  [1/6] Loading data...")
    real_dfs = load_region_data(cfg)

    print("  [2/6] Running baseline benchmark...")
    baseline_hp = {**cfg["initial_hp"], "seeds": RANDOM_SEEDS}
    baseline_bench = benchmark_seeds(real_dfs, baseline_hp, cfg)

    print(f"\n  BASELINE — Per-column:")
    print(f"  {'Column':<20s} {'JS Div':>10} {'WD_norm':>10}")
    print(f"  {'-' * 40}")
    for col in baseline_bench["grand_avg"].index:
        print(f"  {col:<20s} {baseline_bench['grand_avg'].loc[col, 'js_divergence']:>10.4f} "
              f"{baseline_bench['grand_avg'].loc[col, 'wasserstein_norm']:>10.4f}")

    study = None
    hp_checkpoint = load_checkpoint(tag) if skip_if_checkpoint else None

    if hp_checkpoint is not None:
        print("  [3/6] Using checkpoint — skipping Optuna...")
        hp_optimised = {**cfg["initial_hp"], **hp_checkpoint, "seeds": RANDOM_SEEDS}
    else:
        print("  [3/6] Running Optuna optimisation...")
        study = run_optimization(real_dfs, cfg)
        hp_optimised = {**cfg["initial_hp"], **study.best_params, "seeds": RANDOM_SEEDS}

    print("  [4/6] Running full evaluation...")
    results = full_evaluation(real_dfs, hp_optimised, cfg, study)

    # Print summary
    print(f"\n  OPTIMISED — Per-column:")
    ga = results["benchmark"]["grand_avg"]
    print(f"  {'Column':<20s} {'JS Div':>10} {'WD_norm':>10}")
    print(f"  {'-' * 40}")
    for col in ga.index:
        print(f"  {col:<20s} {ga.loc[col, 'js_divergence']:>10.4f} "
              f"{ga.loc[col, 'wasserstein_norm']:>10.4f}")

    # Sanity check: did optimisation actually help? Compare to baseline.
    b = baseline_bench["grand_avg"]
    common = b.index.intersection(ga.index)
    js_worse = sum(1 for c in common if ga.loc[c, "js_divergence"]    > b.loc[c, "js_divergence"])
    wd_worse = sum(1 for c in common if ga.loc[c, "wasserstein_norm"] > b.loc[c, "wasserstein_norm"])
    n_cols = len(common)
    if n_cols > 0 and (js_worse > n_cols / 2 or wd_worse > n_cols / 2):
        print(f"\n  ⚠  WARNING: optimisation produced WORSE fit than baseline "
              f"on {js_worse}/{n_cols} columns (JS) and {wd_worse}/{n_cols} (WD).")
        print(f"     The low best_loss value is likely an optimisation artifact, "
              f"not a fit improvement. Check for NaN metrics, broken columns, "
              f"or weight collapse in softmax_weights_{tag}.json.")

    print(f"\n  SDMetrics Quality:  {results['sdmetrics']['quality_score']:.4f}")
    print(f"  SDMetrics Diagnostic: {results['sdmetrics']['diag_score']:.4f}")
    print(f"  DCR Privacy:        {results['sdmetrics']['dcr_score']:.4f}")
    print(f"  Frobenius Distance: {results['frob_dist']:.4f}")
    print(f"  SWD: {results['multivariate']['swd']:.4f}  "
          f"MMD: {results['multivariate']['mmd']:.6f}")

    # Real-vs-Real vs Real-vs-Synthetic comparison
    rvr_avg = results["real_vs_real"].get("grand_avg")
    if rvr_avg is not None:
        print(f"\n  REAL↔REAL vs REAL↔SYNTHETIC:")
        print(f"  {'Column':<20s} {'JS R↔R':>10} {'JS R↔S':>10} {'Δ':>8}"
              f"  {'WD R↔R':>10} {'WD R↔S':>10} {'Δ':>8}")
        print(f"  {'-' * 78}")
        for col in ga.index:
            if col in rvr_avg.index:
                jr = rvr_avg.loc[col, "js_divergence"]
                js = ga.loc[col, "js_divergence"]
                wr = rvr_avg.loc[col, "wasserstein_norm"]
                ws = ga.loc[col, "wasserstein_norm"]
                print(f"  {col:<20s} {jr:>10.4f} {js:>10.4f} {js-jr:>+8.4f}"
                      f"  {wr:>10.4f} {ws:>10.4f} {ws-wr:>+8.4f}")

    print("  [5/6] Generating visualizations...")
    run_all_visualizations(results, cfg, study)

    wall_time = time.time() - region_t0
    print("  [6/6] Saving checkpoint...")
    if study:
        save_checkpoint(cfg, hp_optimised, results, study,
                        baseline_bench, wall_time)

    # Cleanup
    del real_dfs, results
    plt.close("all")
    gc.collect()

    print(f"\n  {cfg['name']} completed in {int(wall_time // 60)}m {int(wall_time % 60)}s")
    return hp_optimised

## 9.2 Function to run all regiond

In [ ]:
def run_all(regions=None):
    """Run the full pipeline for all (or selected) regions."""
    notebook_start = time.time()
    regions = regions or list(REGION_CONFIGS.keys())

    for tag in regions:
        cfg = REGION_CONFIGS[tag]
        run_region(cfg)

    total = time.time() - notebook_start
    print(f"\n{'═' * 80}")
    print(f"  ALL REGIONS COMPLETE")
    print(f"  Total execution time: {int(total // 60)}m {int(total % 60)}s")
    print(f"{'═' * 80}")

# 10. Running the pipeline

In [ ]:
run_all()